# Assignment 3: RAG Chat System

A simple Retrieval-Augmented Generation (RAG) system that reads uploaded PDF/TXT files,
chunks the text, builds a FAISS vector index, and answers questions using Qwen.

Student ID: 12349031  
Course: 700.390 Advanced Topics in Neurocomputing  



In [1]:
%%capture
!pip install -q transformers torch accelerate sentencepiece faiss-cpu sentence-transformers pypdf
print("Packages installed")

## 1. Imports

In [2]:
import os
import json
import time
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Running in Google Colab
Device: cuda


## 2. Upload PDF Files

Upload the three PDF files when prompted:
- `course_policy.pdf`
- `lab_guide.pdf`
- `faq.pdf`

The system will parse, chunk, and index them automatically.

In [3]:
from google.colab import files

print("Upload your PDF files (course_policy.pdf, lab_guide.pdf, faq.pdf)")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No files were uploaded. Please upload at least one PDF file to continue.")

uploaded_files = {}
for fname, data in uploaded.items():
    with open(fname, "wb") as f:
        f.write(data)
    uploaded_files[fname] = fname

print(f"\nUploaded {len(uploaded_files)} file(s):")
for fname in uploaded_files:
    print(f"  - {fname}")

Upload your PDF files (course_policy.pdf, lab_guide.pdf, faq.pdf)


Saving course_policy.pdf to course_policy.pdf
Saving faq.pdf to faq.pdf
Saving lab_guide.pdf to lab_guide.pdf

Uploaded 3 file(s):
  - course_policy.pdf
  - faq.pdf
  - lab_guide.pdf


## 4. Parse Text from Files

In [4]:
def parse_file(filepath):
    """Extract plain text from TXT, MD, or PDF files."""
    filepath = Path(filepath)
    ext = filepath.suffix.lower()
    if ext in [".txt", ".md"]:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    elif ext == ".pdf":
        try:
            from pypdf import PdfReader
            reader = PdfReader(filepath)
            return "\n".join(page.extract_text() or "" for page in reader.pages)
        except Exception as e:
            print(f"Could not parse {filepath.name}: {e}")
            return ""
    else:
        print(f"Unsupported format: {ext}")
        return ""

documents = {}
for fname, fpath in uploaded_files.items():
    text = parse_file(fpath)
    if text.strip():
        documents[fname] = text
        print(f"Parsed {fname}: {len(text)} characters")
    else:
        print(f"Warning: {fname} produced no text")

print(f"\nTotal documents loaded: {len(documents)}")

Parsed course_policy.pdf: 990 characters
Parsed faq.pdf: 1016 characters
Parsed lab_guide.pdf: 852 characters

Total documents loaded: 3


## 5. Chunk Text with Overlap

Each chunk keeps the original file name as metadata so sources can be cited later.

In [5]:
def chunk_text(text, source_name, chunk_size=500, overlap=100):
    """Split text into overlapping chunks. Each chunk stores its source file name."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append({
                "text": chunk,
                "source": source_name,
                "char_start": start,
                "char_end": min(end, len(text))
            })
        start += chunk_size - overlap
    return chunks

all_chunks = []
for fname, text in documents.items():
    doc_chunks = chunk_text(text, fname)
    all_chunks.extend(doc_chunks)
    print(f"{fname}: {len(doc_chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"\nExample chunk:")
print(f"  Source : {all_chunks[0]['source']}")
print(f"  Text   : {all_chunks[0]['text'][:120]}...")

course_policy.pdf: 3 chunks
faq.pdf: 3 chunks
lab_guide.pdf: 3 chunks

Total chunks: 9

Example chunk:
  Source : course_policy.pdf
  Text   : Course Policy - Advanced Topics in Neurocomputing
Grading
Assignments count for 60% of the final grade. The written exam...


## 6. Embed Chunks and Build FAISS Index

Using `all-MiniLM-L6-v2` (384-dim) to embed all chunks, then building a FAISS flat index.

In [6]:
print("Loading embedding model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks...")
chunk_texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(chunk_texts, batch_size=32, show_progress_bar=True)
embeddings = embeddings.astype(np.float32)

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"\nFAISS index built:")
print(f"  Vectors : {index.ntotal}")
print(f"  Dims    : {dim}")

Loading embedding model (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding all chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


FAISS index built:
  Vectors : 9
  Dims    : 384


## 7. Load Qwen Model

In [7]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print(f"Model loaded on {next(model.parameters()).device}")

Loading Qwen/Qwen3.5-0.8B...


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Model loaded on cuda:0


## 8. RAG Pipeline

**Grounding rule:** The model must answer only from the retrieved chunks.
If the answer is not supported, it must return: *"Insufficient evidence."*

In [8]:
def retrieve(query, k=3):
    """Embed the query and return the top-k most similar chunks."""
    q_emb = embedder.encode([query]).astype(np.float32)
    distances, indices = index.search(q_emb, k)
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        if idx < len(all_chunks):
            chunk = all_chunks[idx].copy()
            chunk["distance"] = float(dist)
            results.append(chunk)
    return results


def generate_answer(query, retrieved_chunks):
    """Build a grounded prompt and generate an answer using Qwen."""
    context = "\n\n".join(
        f"[Source: {c['source']}]\n{c['text']}"
        for c in retrieved_chunks
    )
    prompt = (
        "You are a helpful assistant. Answer the question using ONLY the context below.\n"
        "If the answer cannot be found in the context, respond with exactly: "
        "Insufficient evidence.\n"
        "Do not add any information that is not present in the context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()
    sources = sorted(set(c["source"] for c in retrieved_chunks))
    return response, sources


def rag_query(query, k=3):
    """Full RAG pipeline: retrieve top-k chunks, generate answer, cite sources."""
    start = time.time()
    retrieved = retrieve(query, k)
    answer, sources = generate_answer(query, retrieved)
    elapsed = time.time() - start
    return {
        "question": query,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": retrieved,
        "latency": round(elapsed, 2)
    }

print("RAG pipeline ready.")

RAG pipeline ready.


## 9. Sample Q&A Demo

Three sample questions demonstrating the full pipeline: answer + retrieved chunks + sources.

In [9]:
DEMO_QUESTIONS = [
    "What is the late submission penalty?",
    "Can I submit assignments in groups?",
    "Which GPU is recommended for experiments?"
]

print("=" * 65)
print("SAMPLE Q&A DEMO")
print("=" * 65)

for q in DEMO_QUESTIONS:
    result = rag_query(q)
    print(f"\nQ: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Sources: {', '.join(result['sources'])}")
    print(f"Latency: {result['latency']}s")
    print()
    print("  Retrieved chunks:")
    for i, c in enumerate(result['retrieved_chunks'], 1):
        print(f"  [{i}] {c['source']} | {c['text'][:100]}...")
    print("-" * 65)

SAMPLE Q&A DEMO

Q: What is the late submission penalty?
A: The late submission penalty is 10%.
Sources: course_policy.pdf, faq.pdf
Latency: 2.68s

  Retrieved chunks:
  [1] course_policy.pdf | Course Policy - Advanced Topics in Neurocomputing
Grading
Assignments count for 60% of the final gra...
  [2] course_policy.pdf | udents are expected to attend at least 80% of all lectures. Missing more than 20% of lectures may
re...
  [3] faq.pdf | ory in your report or notebook.
What if my Colab session disconnects during training?
Save checkpoin...
-----------------------------------------------------------------

Q: Can I submit assignments in groups?
A: No. All assignments must be submitted individually. Group discussion is allowed but submitted code and reports must be entirely your own work. Do not share files with other students.
Sources: course_policy.pdf, faq.pdf
Latency: 2.21s

  Retrieved chunks:
  [1] faq.pdf | Frequently Asked Questions
Can I submit assignments in groups?
No. All a

## 10. Evaluation — 10 Test Questions

10 questions covering all required metrics:
- Questions 1–8: answerable from the documents
- Questions 9–10: **not** in the documents → model must return *"Insufficient evidence."*

In [10]:
TEST_QUESTIONS = [
    {"q": "What is the late submission penalty?",               "answerable": True,  "expected_source": "course_policy.txt"},
    {"q": "How is the final grade calculated?",                 "answerable": True,  "expected_source": "course_policy.txt"},
    {"q": "When are the instructor's office hours?",            "answerable": True,  "expected_source": "course_policy.txt"},
    {"q": "How many assignments are in the course?",            "answerable": True,  "expected_source": "course_policy.txt"},
    {"q": "What packages do I need to install for the labs?",   "answerable": True,  "expected_source": "lab_guide.txt"},
    {"q": "Which GPU is recommended for experiments?",          "answerable": True,  "expected_source": "lab_guide.txt"},
    {"q": "Can I submit assignments in groups?",                "answerable": True,  "expected_source": "faq.txt"},
    {"q": "Can I use pre-trained models from Hugging Face?",    "answerable": True,  "expected_source": "faq.txt"},
    {"q": "What is the capital of Germany?",                    "answerable": False, "expected_source": None},
    {"q": "How does BERT compute self-attention?",              "answerable": False, "expected_source": None},
]

eval_results = []
print("Running evaluation on 10 test questions...\n")

for i, item in enumerate(TEST_QUESTIONS, 1):
    result = rag_query(item["q"])

    # Retrieval quality: expected source in top-k chunks
    retrieved_sources = [c["source"] for c in result["retrieved_chunks"]]
    retrieval_hit = (
        item["expected_source"] in retrieved_sources
        if item["expected_source"] else True
    )

    # Abstention: model should say insufficient evidence for unanswerable questions
    abstained = "insufficient evidence" in result["answer"].lower()
    abstention_ok = (
        (not item["answerable"] and abstained) or
        (item["answerable"] and not abstained)
    )

    # Grounding: simple heuristic - check if answer words appear in retrieved chunks
    if item["answerable"] and not abstained:
        answer_words = set(result["answer"].lower().split())
        context_words = set(" ".join(c["text"] for c in result["retrieved_chunks"]).lower().split())
        overlap = len(answer_words & context_words) / max(len(answer_words), 1)
        grounded = overlap > 0.4
    else:
        grounded = None

    eval_results.append({
        "id": i,
        "question": item["q"],
        "answer": result["answer"],
        "sources_cited": result["sources"],
        "retrieved_sources": retrieved_sources,
        "expected_source": item["expected_source"],
        "retrieval_hit": retrieval_hit,
        "grounded": grounded,
        "abstained": abstained,
        "abstention_ok": abstention_ok,
        "answerable": item["answerable"],
        "latency": result["latency"]
    })

    status = "OK" if abstention_ok else "FAIL"
    print(f"[Q{i}] [{status}] {item['q']}")
    print(f"      A: {result['answer'][:100]}")
    print(f"      Sources: {result['sources']} | {result['latency']}s")
    print()

print("Evaluation complete.")

Running evaluation on 10 test questions...

[Q1] [OK] What is the late submission penalty?
      A: The late submission penalty is 10%.
      Sources: ['course_policy.pdf', 'faq.pdf'] | 0.79s

[Q2] [OK] How is the final grade calculated?
      A: The final grade is calculated as follows:
- Assignments count for 60% of the final grade.
- The writ
      Sources: ['course_policy.pdf'] | 7.22s

[Q3] [OK] When are the instructor's office hours?
      A: Monday from 2pm to 4pm and on Wednesday from 10am to 12pm.
      Sources: ['course_policy.pdf', 'faq.pdf'] | 1.74s

[Q4] [OK] How many assignments are in the course?
      A: Based on the context provided, there are 4 assignments in total.
      Sources: ['course_policy.pdf'] | 1.29s

[Q5] [OK] What packages do I need to install for the labs?
      A: Install the following packages: transformers, torch, accelerate, sentencepiece, faiss-cpu, sentence-
      Sources: ['course_policy.pdf', 'lab_guide.pdf'] | 1.86s

[Q6] [OK] Which GPU is recomm

## 11. Evaluation Summary Table

In [11]:
print("=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)
header = f"{'#':<3} {'Question':<42} {'Retrieval':<10} {'Grounded':<10} {'Abstain':<8} {'Time':>6}"
print(header)
print("-" * 80)

for r in eval_results:
    retrieval  = "Yes" if r["retrieval_hit"]  else "No"
    grounded   = "Yes" if r["grounded"] is True else ("No" if r["grounded"] is False else "N/A")
    abstain    = "N/A" if r["answerable"] else ("OK" if r["abstention_ok"] else "FAIL")
    q_short    = r["question"][:41]
    print(f"{r['id']:<3} {q_short:<42} {retrieval:<10} {grounded:<10} {abstain:<8} {r['latency']:>5.2f}s")

print("=" * 80)

n          = len(eval_results)
ret_hits   = sum(1 for r in eval_results if r["retrieval_hit"])
grnd_hits  = sum(1 for r in eval_results if r["grounded"] is True)
grnd_total = sum(1 for r in eval_results if r["grounded"] is not None)
abst_ok    = sum(1 for r in eval_results if not r["answerable"] and r["abstention_ok"])
avg_lat    = sum(r["latency"] for r in eval_results) / n

print(f"\nRetrieval quality   : {ret_hits}/{n} correct source in top-k chunks")
print(f"Grounding           : {grnd_hits}/{grnd_total} answers supported by retrieved chunks (heuristic)")
print(f"Source correctness  : cited file name matches expected source")
print(f"Abstention          : {abst_ok}/2 unanswerable questions returned Insufficient evidence.")
print(f"Average latency     : {avg_lat:.2f}s per question")

# Save evaluation results
os.makedirs("results", exist_ok=True)
with open("results/evaluation_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("\nSaved to results/evaluation_results.json")

EVALUATION SUMMARY
#   Question                                   Retrieval  Grounded   Abstain    Time
--------------------------------------------------------------------------------
1   What is the late submission penalty?       No         Yes        N/A       0.79s
2   How is the final grade calculated?         No         Yes        N/A       7.22s
3   When are the instructor's office hours?    No         Yes        N/A       1.74s
4   How many assignments are in the course?    No         Yes        N/A       1.29s
5   What packages do I need to install for th  No         Yes        N/A       1.86s
6   Which GPU is recommended for experiments?  No         Yes        N/A       1.02s
7   Can I submit assignments in groups?        No         Yes        N/A       1.93s
8   Can I use pre-trained models from Hugging  No         Yes        N/A       1.97s
9   What is the capital of Germany?            Yes        N/A        FAIL      0.65s
10  How does BERT compute self-attention?      Yes

## 12. Download Results

In [12]:
if IN_COLAB:
    import shutil
    shutil.make_archive("assignment_3_results", "zip", "results")
    from google.colab import files
    files.download("assignment_3_results.zip")
    print("Downloaded assignment_3_results.zip")
else:
    print("Results saved in results/ folder")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded assignment_3_results.zip
